# Getting MGnify data

The [MGnify API](https://www.ebi.ac.uk/metagenomics/api/v2/) provides access to MGnify analyses datasets and important metadata such as biome, sample, study, run, analysis details. On this page we demonstrate how to:

- **Get** the metadata using `MGnifier`
- **Get** the datasets as a `MGazine`

```{margin}
After clicking the "Activate Notebook" button you can run the cells in this browser. Alternatively, you can also click on the 🚀 to launch in colab or binder.
```
<button title="Make live" style="display:inline-flex;align-items:center;gap:0.4rem;padding:0.5rem 1rem;border:0;border-radius:20px;background:linear-gradient(135deg,#0f766e,#14b8a6);color:white;cursor:pointer;font-size:1rem;" class="thebe-button" onclick="initThebeSBT()">Activate Notebook</button>

---

In [1]:
# uncomment below if colab
#!pip install mgnipy

Recall the typical workflow (from [What is MGni.Py?](https://mgnipy.mgnify.org/notebooks/getting-started/1_what_is_mgnipy.html)):

> 1. Start up a `mgnipy.MGnipy` client with your desired configuration
> 
> 2. Search in MGnify resources using a MGnifier glass
> 
> 3. Receive a MGazine of MGnify datasets

which we will follow in this notebook

## 1. `mgnipy.MGnipy` to init session

In [2]:
from mgnipy import MGnipy

MG = MGnipy(
    cache_dir=None
)

## 2. `MGnifier` to query MGnify

for this example we will search MGnify `.studies` for a list of pea studies. 

After we `.get()` the list we will populate each of the study's details / metadata using `.enrich_details()`

In [3]:
# access studies MGnifier and pass search params (build query set)
pea_studies = MG.studies(
    biome_lineage="root:Host-associated:Plants", 
    search="pea"
)

# check out the request url
pea_studies.explain()

https://www.ebi.ac.uk/metagenomics/api/v2/studies?biome_lineage=root%3AHost-associated%3APlants&search=pea&page=1


now executing the query url(s). If there were multiple urls in the query set then we could also use `.get_all()` rather than iteratively `.get`ting page by page.

In [4]:
# MG as client context manager
with MG: 
    # get a page of pea study list
    pea_studies.get()
    # now filling with metadata
    pea_studies.enrich_details()

Enriching study details: 100%|██████████| 4/4 [00:00<00:00, 14.96it/s]


We can access the detailed metadata via `.metadata` attribute which will return a [`MGnifyMetadata`](TODO) instance that allows you to view as a list, polars or pandas dataframe. 

In [5]:
# accessing metadata
meta = pea_studies.metadata

# as pandas dataframe
meta.to_pandas(expand_nested_dicts=True)

,accession,ena_accessions,title,updated_at,downloads,first_accession,biome__biome_name,biome__lineage
0,MGYS00010230,"[SRP288829, PRJNA670831]",The microbial community associated with pea se...,2026-04-21T08:55:57.244000+00:00,"[{'file_type': 'tsv', 'download_type': 'Taxono...",SRP288829,Plants,root:Host-associated:Plants
1,MGYS00000829,"[SRP029898, PRJNA216952]",Wetland peat Targeted Locus (Loci),2026-05-28T15:46:50.398000+00:00,"[{'file_type': 'tsv', 'download_type': 'Taxono...",SRP029898,Rhizosphere,root:Host-associated:Plants:Rhizosphere
2,MGYS00000399,"[ERP006378, PRJEB6752]",Transcriptomic analysis of microbial communiti...,2026-05-28T15:46:49.171000+00:00,"[{'file_type': 'tsv', 'download_type': 'Taxono...",ERP006378,Soil,root:Host-associated:Plants:Rhizosphere:Soil
3,MGYS00001374,"[ERP014435, PRJEB12905]",Ectomycorrhizal and fine root foraging across ...,2026-05-28T15:46:51.301000+00:00,"[{'file_type': 'tsv', 'download_type': 'Taxono...",ERP014435,Forest soil,root:Host-associated:Plants:Rhizosphere:Forest...


## 3. `MGazine` of MGnify datasets

To access the study's mgazine use `.datasets`

- Notice how in study details printed above there is a "downloads" field with information about the data. 

- this "downloads" information is used by [`mgnipy.MGazine`](TODO) to allow us to download or read them into our notebook.

- To access the study's mgazine use `.datasets`

- the __str__ representaiton of mgazine gives us a peak into the pipeline versions within, number of downloads and the short description categories

In [6]:
# access study mgazine
MZ = pea_studies.datasets

# print for more info
print(MZ)

# also can view more as df
MZ.downloads_df()

MGazine containing:
- MGnify pipeline versions: ['v1', 'v2', 'v3', 'v6']
- Number of downloads: 20
- Short descriptions: ['Complete GO annotation',
 'GO slim annotation',
 'InterPro matches',
 'PCA for runs (based on phylum proportions)',
 'Phylum level taxonomies',
 'Summary of DADA2-PR2 taxonomies',
 'Summary of DADA2-SILVA taxonomies',
 'Summary of ITSoneDB taxonomies',
 'Summary of PR2 taxonomies',
 'Summary of SILVA-LSU taxonomies',
 'Summary of SILVA-SSU taxonomies',
 'Summary of UNITE taxonomies',
 'Taxonomic assignments']
- Nonempty metadata sets: .mgnify_studies



,file_type,download_type,short_description,long_description,alias,download_group,file_size_bytes,index_files,url,accession,pipeline_version
0,tsv,Taxonomic analysis,Summary of UNITE taxonomies,"Summary of UNITE taxonomic assignments, across...",SRP288829_UNITE_study_summary.tsv,study_summary.v6.amplicon,None,None,https://ftp.ebi.ac.uk/pub/databases/metagenomi...,MGYS00010230,v6
1,tsv,Taxonomic analysis,Summary of SILVA-LSU taxonomies,"Summary of SILVA-LSU taxonomic assignments, ac...",SRP288829_SILVA-LSU_study_summary.tsv,study_summary.v6.amplicon,None,None,https://ftp.ebi.ac.uk/pub/databases/metagenomi...,MGYS00010230,v6
2,tsv,Taxonomic analysis,Summary of SILVA-SSU taxonomies,"Summary of SILVA-SSU taxonomic assignments, ac...",SRP288829_SILVA-SSU_study_summary.tsv,study_summary.v6.amplicon,None,None,https://ftp.ebi.ac.uk/pub/databases/metagenomi...,MGYS00010230,v6
3,tsv,Taxonomic analysis,Summary of ITSoneDB taxonomies,"Summary of ITSoneDB taxonomic assignments, acr...",SRP288829_ITSoneDB_study_summary.tsv,study_summary.v6.amplicon,None,None,https://ftp.ebi.ac.uk/pub/databases/metagenomi...,MGYS00010230,v6
4,tsv,Taxonomic analysis,Summary of DADA2-PR2 taxonomies,"Summary of DADA2-PR2 taxonomic assignments, ac...",SRP288829_DADA2-PR2_16S-V3-V4_study_summary.tsv,study_summary.v6.amplicon,None,None,https://ftp.ebi.ac.uk/pub/databases/metagenomi...,MGYS00010230,v6
5,tsv,Taxonomic analysis,Summary of DADA2-SILVA taxonomies,"Summary of DADA2-SILVA taxonomic assignments, ...",SRP288829_DADA2-SILVA_16S-V3-V4_study_summary.tsv,study_summary.v6.amplicon,None,None,https://ftp.ebi.ac.uk/pub/databases/metagenomi...,MGYS00010230,v6
6,tsv,Taxonomic analysis,Summary of PR2 taxonomies,"Summary of PR2 taxonomic assignments, across a...",SRP288829_PR2_study_summary.tsv,study_summary.v6.amplicon,None,None,https://ftp.ebi.ac.uk/pub/databases/metagenomi...,MGYS00010230,v6
7,tsv,Taxonomic analysis,Phylum level taxonomies,Phylum level taxonomies (TSV),SRP029898_phylum_taxonomy_abundances_v2.0.tsv,study_summary.v2.0.taxonomic_analysis,None,None,https://ftp.ebi.ac.uk/pub/databases/metagenomi...,MGYS00000829,v2
8,tsv,Taxonomic analysis,Taxonomic assignments,Taxonomic assignments (TSV),SRP029898_taxonomy_abundances_v2.0.tsv,study_summary.v2.0.taxonomic_analysis,None,None,https://ftp.ebi.ac.uk/pub/databases/metagenomi...,MGYS00000829,v2
9,tsv,Taxonomic analysis,Phylum level taxonomies,Phylum level taxonomies (TSV),ERP006378_phylum_taxonomy_abundances_v1.0.tsv,study_summary.v1.0.taxonomic_analysis,None,None,https://ftp.ebi.ac.uk/pub/databases/metagenomi...,MGYS00000399,v1


You can read in whole or stream in chunks a dataset by passing its `alias` or `url` to `MGazine.stream()`

In [10]:
alias = "ERP014435_GO-slim_abundances_v3.0.tsv"
# reading in above file
df_go = MZ.stream(
    alias=alias,
    chunksize=None, # default to read in all, set int for chunked reading
    df_engine="pandas", # or polars
)

df_go.head()

,GO,description,category,ERR1299314,ERR1299315,ERR1299316,ERR1299317,ERR1299319,ERR1299320,ERR1299321,ERR1299323,ERR1299324,ERR1299325,ERR1299326,ERR1299330,ERR1299331,ERR1299333
0,GO:0000015,phosphopyruvate hydratase complex,cellular component,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,GO:0000150,recombinase activity,molecular function,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,GO:0000160,phosphorelay signal transduction system,biological process,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,GO:0000166,nucleotide binding,molecular function,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,GO:0003674,molecular function,molecular function,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [16]:
# run accessions as a list
run_ids = df_go.columns[3:].to_list()
#check it out
print(run_ids)

['ERR1299314', 'ERR1299315', 'ERR1299316', 'ERR1299317', 'ERR1299319', 'ERR1299320', 'ERR1299321', 'ERR1299323', 'ERR1299324', 'ERR1299325', 'ERR1299326', 'ERR1299330', 'ERR1299331', 'ERR1299333']


you can also `.download()` or `.download_all()` of the files to a directory of your choosing

In [9]:
MZ.download(alias=alias, to_dir="downloads")

---

## Wrap Up:

This page was a quick start demonstration of:

1. ✅ Start up a `mgnipy.MGnipy` client with your desired configuration

2. ✅ Querying MGnify using a `MGnifier` glass

3. ✅ Accessing the resulting `MGazine` of MGnify datasets

**Next** we will see how we can collect even more metadata for the above list of `run_ids` using  mgnipy's `MGnetizer` and `BioSampler` helpers. 